In [0]:

# Configuracion
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

CATALOG = "banco"
SCHEMA = "banking_analytics"

# Tablas de origen
SILVER_TABLES = {
    "branches": f"{CATALOG}.{SCHEMA}.silver_branches",
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "accounts": f"{CATALOG}.{SCHEMA}.silver_accounts",
    "transactions": f"{CATALOG}.{SCHEMA}.silver_transactions"
}

# Tablas destino Gold
GOLD_TABLES = {
    "dim_customer": f"{CATALOG}.{SCHEMA}.dim_customer",
    "dim_account": f"{CATALOG}.{SCHEMA}.dim_account",
    "dim_branch": f"{CATALOG}.{SCHEMA}.dim_branch",
    "dim_date": f"{CATALOG}.{SCHEMA}.dim_date",
    "fact_transactions": f"{CATALOG}.{SCHEMA}.fact_transactions"
}


In [0]:
# Cargar tablas silver
df_silver_branches = spark.table(SILVER_TABLES["branches"])

df_silver_customers = spark.table(SILVER_TABLES["customers"])

df_silver_accounts = spark.table(SILVER_TABLES["accounts"])

df_silver_transactions = spark.table(SILVER_TABLES["transactions"])

print("Branches:", df_silver_branches.count())
print("Customers:", df_silver_customers.count())
print("Accounts:", df_silver_accounts.count())
print("Transactions:", df_silver_transactions.count())


In [0]:
# Verificación de Schema
print("CUSTOMERS")
df_silver_customers.printSchema()

print("ACCOUNTS")
df_silver_accounts.printSchema()

print("BRANCHES")
df_silver_branches.printSchema()

print("TRANSACTIONS")
df_silver_transactions.printSchema()


In [0]:
# Construccion de la dimension Customer

df_dim_customer = (
    df_silver_customers

    .select(
        "customer_id",
        "customer_name",
        "birth_date",
        "country",
        "registration_date",
        "email",
        "customer_segment"
    )

    # Generar clave sustituta determinista
    .withColumn(
        "customer_key",
        F.xxhash64(F.col("customer_id"))
    )

    # Organizar las columnas
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "birth_date",
        "country",
        "registration_date",
        "email",
        "customer_segment"
    )
)

display(df_dim_customer.limit(10))


In [0]:

# Validar la dimension Customer

total_silver = df_silver_customers.count()

total_dimension = df_dim_customer.count()

duplicate_customers = (
    df_dim_customer
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_customer
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Clientes Silver:", total_silver)
print("Clientes DIM:", total_dimension)
print("Customer ID duplicados:", duplicate_customers)
print("Customer Key duplicados:", duplicate_keys)

assert total_silver == total_dimension
assert duplicate_customers == 0
assert duplicate_keys == 0

print("Validaciones de DIM_CUSTOMER exitosas.")


In [0]:
# GUARDAR TABLA DIMENSION CUSTOMER
(
    df_dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_customer"])
)

In [0]:
# Validar persistencia de datos
df_dim_customer_saved = spark.table(
    GOLD_TABLES["dim_customer"]
)

print("Registros guardados:", df_dim_customer_saved.count())

display(
    df_dim_customer_saved.limit(10)
)

In [0]:
# Construcción de Dimen Branch
df_dim_branch = (
    df_silver_branches
    .select(
        "branch_id",
        "branch_name",
        "city",
        "region",
        "country",
        "opening_date",
        "is_active"
    )
    .withColumn(
        "branch_key",
        F.xxhash64(F.col("branch_id"))
    )
    .select(
        "branch_key",
        "branch_id",
        "branch_name",
        "city",
        "region",
        "country",
        "opening_date",
        "is_active"
    )
)

display(df_dim_branch.limit(10))

In [0]:
# Validar Integridad de datos
total_silver = df_silver_branches.count()
total_dimension = df_dim_branch.count()

duplicate_branches = (
    df_dim_branch
    .groupBy("branch_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_branch
    .groupBy("branch_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Agencias Silver:", total_silver)
print("Agencias DIM:", total_dimension)
print("Branch ID duplicados:", duplicate_branches)
print("Branch Key duplicados:", duplicate_keys)

assert total_silver == total_dimension
assert duplicate_branches == 0
assert duplicate_keys == 0

print("Validaciones de DIM_BRANCH exitosas.")

In [0]:
#Persistir tabla
(
    df_dim_branch.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_branch"])
)

In [0]:
# Validar persistencia
df_dim_branch_saved = spark.table(
    GOLD_TABLES["dim_branch"]
)

print(
    "Registros guardados:",
    df_dim_branch_saved.count()
)

display(df_dim_branch_saved.limit(10))

In [0]:
# cargar dimensiones
df_dim_customer = spark.table(
    GOLD_TABLES["dim_customer"]
)

df_dim_branch = spark.table(
    GOLD_TABLES["dim_branch"]
)

In [0]:
# Construir la dimension mediante joins

df_dim_account = (
    df_silver_accounts.alias("a")

    .join(
        df_dim_customer.select(
            "customer_id",
            "customer_key"
        ).alias("c"),
        F.col("a.customer_id") == F.col("c.customer_id"),
        "inner"
    )

    .join(
        df_dim_branch.select(
            "branch_id",
            "branch_key"
        ).alias("b"),
        F.col("a.branch_id") == F.col("b.branch_id"),
        "inner"
    )

    .select(
        F.xxhash64(F.col("a.account_id")).alias("account_key"),
        F.col("a.account_id"),
        F.col("c.customer_key"),
        F.col("b.branch_key"),
        F.col("a.account_type"),
        F.col("a.balance"),
        F.col("a.currency"),
        F.col("a.opening_date"),
        F.col("a.account_status")
    )
)

display(df_dim_account.limit(10))

In [0]:
# Validacion de integridad de datos

total_silver = df_silver_accounts.count()
total_dimension = df_dim_account.count()

duplicate_accounts = (
    df_dim_account
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_account
    .groupBy("account_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_keys = (
    df_dim_account
    .filter(F.col("customer_key").isNull())
    .count()
)

null_branch_keys = (
    df_dim_account
    .filter(F.col("branch_key").isNull())
    .count()
)

print("Cuentas Silver:", total_silver)
print("Cuentas DIM:", total_dimension)
print("Account ID duplicados:", duplicate_accounts)
print("Account Key duplicados:", duplicate_keys)
print("Customer Key nulos:", null_customer_keys)
print("Branch Key nulos:", null_branch_keys)

In [0]:
# Persistencia de datos
(
    df_dim_account.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_account"])
)

In [0]:
# Validar persistencia de datos
df_dim_account_saved = spark.table(
    GOLD_TABLES["dim_account"]
)

print(
    "Registros guardados:",
    df_dim_account_saved.count()
)

display(df_dim_account_saved.limit(10))